# example / wasm / saturate

Squash each input through a fitted `tanh` curve, as a WebAssembly module that binds the host's `tanh`.

| Term | In This Notebook |
|---|---|
| **Intent** | Show a `wasm` model that imports a host maths function end to end: fit, build the module, accept, replay the golden within a tolerance and register |
| **Project Flavour** | `wasm` (`model.md` 3.2) |
| **Data Splits** | `training` ten labelled points on a curve with fixed noise, to fit; `validation` five more; `backtest` ten further out, held back for acceptance; `golden` the exact samples that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
import json
import math
from os.path import join

from scipy.optimize import curve_fit

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir
from workbench.wasm import build_wasm, tanh, trace

env = init_env()

WORKSPACE = "example/wasm/saturate"
MODEL_NAME = "example_wasm_saturate"
TARGETS = [{"kind": "served"}, {"kind": "embedded", "language": "python"}, {"kind": "embedded", "language": "go"}, {"kind": "embedded", "language": "rust"}]
TRAINING = "wasm_saturate_training_2026-11-03_2026-11-03"
VALIDATION = "wasm_saturate_validation_2026-11-03_2026-11-03"
BACKTEST = "wasm_saturate_backtest_2026-11-03_2026-11-03"
GOLDEN = "wasm_saturate_golden"
INPUT = "example_saturate_in"
OUTPUT = "example_saturate_out"

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, so no query runs.

In [ ]:
NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [first + 0.5 * step for step in range(count)]
    return {INPUT: xs, OUTPUT: [round(4.0 * math.tanh(0.5 * x) + NOISE[int(round(2 * x)) % 10], 4) for x in xs]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled(-2.5, 10))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled(2.5, 5))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled(5.0, 10))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit a saturating curve, height times `tanh` of rate times input, on `training` only (R6). The curve is one Python function, `saturate`, which `curve_fit` calls on arrays here and `workbench.wasm.trace` turns into the module below (`model.md` 4.4). Its height and rate are rounded to one place, which the module builds in.

In [ ]:
def saturate(x, height, rate):
    return height * tanh(rate * x)


(height, rate), _ = curve_fit(saturate, training_pl[INPUT].to_numpy(), training_pl[OUTPUT].to_numpy(), p0=(3.0, 0.4))
HEIGHT, RATE = round(float(height), 1), round(float(rate), 1)
print("fitted height", HEIGHT, "rate", RATE)
assert (HEIGHT, RATE) == (4.0, 0.5), "the fit did not recover the curve that made the data"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** trace the fitted `saturate` into the module, which imports `tanh` from the host (`model.md` 4.1), build the signature, run the module through the library host and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output
from workbench.replay import model_predict

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.wasm")
with open(model_path, "wb") as model_file:
    model_file.write(build_wasm(1, trace(lambda x: saturate(x, HEIGHT, RATE), 1)))

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "A number to squash.")],
    [signature_output(OUTPUT, "none", "The fitted height times the hyperbolic tangent of the fitted rate times the input.", datatype="FP64")],
)
predict = model_predict(model_path, "wasm", signature)

In [ ]:
def acceptance_checks():
    checks = []
    for split, frame in (("validation", validation_pl), ("backtest", backtest_pl)):
        predicted = [predict({INPUT: [x]})[OUTPUT][0] for x in frame[INPUT].to_list()]
        checks.append({"split": split, "output": OUTPUT, "name": "rmse", "value": measure(frame[OUTPUT].to_list(), predicted)["rmse"], "limit": 0.5, "direction": "max"})
    return checks


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "limit", entry["limit"], "passed", entry["passed"])

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file with a tolerance for the output that depends on the imported `tanh` (`model.md` 4.2), copy the module and the evaluation into a project, verify it and replay the golden through the library host (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [-3.0, -1.0, 0.0, 0.5, 1.0, 2.0, 6.0]
    return {INPUT: [*xs, None], OUTPUT: [*[saturate(x, HEIGHT, RATE) for x in xs], None], "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "wasm",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic curve, training split",
        [training_path, golden_path],
        "least squares tanh curve, height and rate built into the module",
        implementation="WebAssembly traced from the fitted Python function by workbench.wasm.trace and built by build_wasm, importing tanh from asystem_math",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across zero, both signs and the saturated tail",
    tolerance={OUTPUT: {"absolute": 1e-12}},
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)